In [1]:
# ─────────────────────────────────────────────────────────
# Install all required packages
# langchain==1.3.0 | langchain-openai==1.2.1 | langchain-community==0.4.1
# ─────────────────────────────────────────────────────────
%pip install -q langchain==1.3.0
%pip install -q langchain-openai==1.2.1
%pip install -q langchain-community==0.4.1
%pip install -q faiss-cpu
%pip install -q openai
%pip install -q gradio
%pip install -q pypdf                  # PDF loading
%pip install -q python-docx            # Word doc loading
%pip install -q beautifulsoup4 lxml    # HTML loading
%pip install -q Pillow                 # Image handling
%pip install -q python-dotenv         # .env support
%pip install -q docx    
%pip install docx2txt
%pip install langchain-text-splitters
print("✅ All packages installed. Please restart the kernel before proceeding.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.


...

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.
  Using cached docx2txt-0.9-py3-none-any.whl.metadata (529 bytes)
Using cached docx2txt-0.9-py3-none-any.whl (4.0 kB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✅ All packages installed. Please restart the kernel before proceeding.


In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# ── Load API key from .env file (recommended) OR set directly ──
load_dotenv()  # reads OPENAI_API_KEY from .env file in current directory

# If you don't have a .env file, uncomment and set directly:
os.environ["OPENAI_API_KEY"] = ""

In [3]:
SOURCE_FOLDER = Path("source_docs")       # Put your PDFs, DOCX, HTML here
print(SOURCE_FOLDER)

source_docs


In [4]:
FAISS_INDEX_PATH = Path("faiss_index")    # FAISS vector store will be saved here

In [5]:
IMAGE_CACHE_PATH = Path("image_cache")    # Extracted images will be saved here

In [6]:
for folder in [SOURCE_FOLDER, FAISS_INDEX_PATH, IMAGE_CACHE_PATH]:
    folder.mkdir(exist_ok=True)

print(f"📁 Source folder: {SOURCE_FOLDER.resolve()}")
print(f"📁 FAISS index folder: {FAISS_INDEX_PATH.resolve()}")
print(f"📁 Image cache folder: {IMAGE_CACHE_PATH.resolve()}")
print()

📁 Source folder: D:\Training\agentic_ai_training\Section2\HandsOn\source_docs
📁 FAISS index folder: D:\Training\agentic_ai_training\Section2\HandsOn\faiss_index
📁 Image cache folder: D:\Training\agentic_ai_training\Section2\HandsOn\image_cache



In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# ── LLM Wrapper: ChatOpenAI ──────────────────────────────────────────────
# model: gpt-4o-mini is cost-effective for a student chatbot
# temperature=0 → deterministic, factual answers (ideal for exam prep)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [8]:
# ── Vision LLM: GPT-4o for image understanding ────────────────────────────
# Used to summarize concept map images extracted from DOCX files
vision_llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=500
)

In [9]:
# ── Quick Test: Direct LLM call ───────────────────────────────────────────
messages = [
    SystemMessage(content=(
        "You are a helpful tutor for 10th Standard Tamil Nadu State Board students. "
        "Answer in simple, clear language suitable for a 15-year-old student."
    )),
    HumanMessage(content="What is photosynthesis? Give a one-line definition.")
]


In [10]:
response = llm.invoke(messages)
print("🤖 LLM Direct Response (no RAG yet):")
print(response.content)
print()
print(f"📊 Model: {response.response_metadata.get('model_name', 'gpt-4o-mini')}")
print(f"📊 Input tokens: {response.usage_metadata.get('input_tokens', 'N/A')}")
print(f"📊 Output tokens: {response.usage_metadata.get('output_tokens', 'N/A')}")

🤖 LLM Direct Response (no RAG yet):
Photosynthesis is the process by which green plants use sunlight to convert carbon dioxide and water into glucose and oxygen.

📊 Model: gpt-4o-mini-2024-07-18
📊 Input tokens: 53
📊 Output tokens: 22


In [11]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    BSHTMLLoader
)
from langchain_core.documents import Document
import zipfile
import shutil
from PIL import Image
import io

In [12]:
all_documents = []   # will hold all loaded text Documents

In [13]:
pdf_files   = list(SOURCE_FOLDER.glob("*.pdf"))
docx_files  = list(SOURCE_FOLDER.glob("*.docx"))
html_files  = list(SOURCE_FOLDER.glob("*.html"))

In [14]:
print(pdf_files)

[WindowsPath('source_docs/science_chapter3.pdf')]


In [15]:
print(f"📄 Found {len(pdf_files)} PDF(s), {len(docx_files)} DOCX(s), {len(html_files)} HTML(s)")

📄 Found 1 PDF(s), 1 DOCX(s), 2 HTML(s)


In [17]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [18]:
# ── Load PDFs ────────────────────────────────────────────────────────────
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_type"] = "pdf"
        doc.metadata["file_name"]   = pdf_path.name
    all_documents.extend(docs)
    print(f"✅ PDF loaded: {pdf_path.name} → {len(docs)} page(s)")

✅ PDF loaded: science_chapter3.pdf → 14 page(s)


In [19]:
print(all_documents)

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2019-04-01T17:20:00+05:30', 'author': "BYJU'S", 'keywords': 'Tamilnadu Board Class 10 Science Chapter 12', 'moddate': '2019-04-01T17:20:30+05:30', 'subject': 'Tamilnadu Board Class 10 Science Chapter 12', 'title': 'Tamilnadu Board Class 10 Science Chapter 12', 'source': 'source_docs\\science_chapter3.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'file_name': 'science_chapter3.pdf'}, page_content='173\n Introduction \nPlants exhibits varying degrees of \norganization. Atoms are organized into \nmolecules, molecules into organelles, organelles \ninto cells, cells into tissues and tissues into \norgans. The first account of internal structure \nof plants was published by English Physician \nNehemiah Grew. He is known as Father of \nPlant Anatomy. Plant anatomy (Gk Ana = as \nunder; T emnein = to cut) is the study of internal \nstructure o

In [20]:
# ── Load DOCX (text) ─────────────────────────────────────────────────────
for docx_path in docx_files:
    loader = Docx2txtLoader(str(docx_path))
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_type"] = "docx"
        doc.metadata["file_name"]   = docx_path.name
    all_documents.extend(docs)
    print(f"✅ DOCX loaded: {docx_path.name} → {len(docs)} chunk(s)")

✅ DOCX loaded: history_notes.docx → 1 chunk(s)


In [22]:
%pip install beautifulsoup4

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)

   ---------------------------------------- 0/2 [soupsieve]
   ---------------------------------------- 0/2 [soupsieve]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   ---------------------------------------- 2/2 [beautifulsoup4]

Note: you may need to restart the kernel to use updated packages.


In [23]:
# ── Load HTML ─────────────────────────────────────────────────────────────
for html_path in html_files:
    loader = BSHTMLLoader(str(html_path), open_encoding="utf-8")
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_type"] = "html"
        doc.metadata["file_name"]   = html_path.name
    all_documents.extend(docs)
    print(f"✅ HTML loaded: {html_path.name} → {len(docs)} page(s)")

✅ HTML loaded: maths_revision.html → 1 page(s)
✅ HTML loaded: science_model_question_paper.html → 1 page(s)


In [24]:
print(f"📚 Total raw documents loaded: {len(all_documents)}")
print()
print("🔍 Sample document preview:")
if all_documents:
    sample = all_documents[0]
    print(f"   Metadata: {sample.metadata}")
    print(f"   Content snippet: {sample.page_content[:200]}...")

📚 Total raw documents loaded: 17

🔍 Sample document preview:
   Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2019-04-01T17:20:00+05:30', 'author': "BYJU'S", 'keywords': 'Tamilnadu Board Class 10 Science Chapter 12', 'moddate': '2019-04-01T17:20:30+05:30', 'subject': 'Tamilnadu Board Class 10 Science Chapter 12', 'title': 'Tamilnadu Board Class 10 Science Chapter 12', 'source': 'source_docs\\science_chapter3.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'file_name': 'science_chapter3.pdf'}
   Content snippet: 173
 Introduction 
Plants exhibits varying degrees of 
organization. Atoms are organized into 
molecules, molecules into organelles, organelles 
into cells, cells into tissues and tissues into 
organs...


In [26]:
import base64
from langchain_core.messages import HumanMessage


def extract_images_from_docx(docx_path: Path, output_dir: Path) -> list:
    """Extract all images embedded in a .docx file."""
    extracted = []
    output_dir.mkdir(exist_ok=True)
    
    # DOCX files are ZIP archives — images are in word/media/
    with zipfile.ZipFile(str(docx_path), 'r') as z:
        media_files = [f for f in z.namelist() if f.startswith('word/media/')]
        for media_file in media_files:
            ext = Path(media_file).suffix.lower()
            if ext in ['.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff']:
                image_data = z.read(media_file)
                image_name = f"{docx_path.stem}_{Path(media_file).name}"
                save_path  = output_dir / image_name
                with open(save_path, 'wb') as f:
                    f.write(image_data)
                extracted.append({
                    "path": save_path,
                    "source_docx": docx_path.name,
                    "image_name": image_name
                })
    return extracted

In [27]:
def summarize_image_with_vision(image_path: Path, vision_llm, source_file: str) -> str:
    """Send image to GPT-4o Vision and get an educational summary."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    
    # Determine MIME type
    ext = image_path.suffix.lower().lstrip('.')
    mime_map = {'jpg': 'jpeg', 'jpeg': 'jpeg', 'png': 'png', 'gif': 'gif', 'bmp': 'bmp'}
    mime_type = f"image/{mime_map.get(ext, 'png')}"
    
    b64_image = base64.b64encode(image_bytes).decode('utf-8')
    
    message = HumanMessage(content=[
        {
            "type": "text",
            "text": (
                "You are analyzing an educational image from a 10th Standard Tamil Nadu State Board "
                "study material. This image is from the file: " + source_file + ".\n\n"
                "Please provide a detailed educational description of this image. Include:\n"
                "1. What concept or topic does this image represent?\n"
                "2. All labels, text, or annotations visible in the image\n"
                "3. The relationships shown (arrows, connections, hierarchy)\n"
                "4. How a student should interpret this for their exam\n"
                "Describe it as if explaining to a 10th std student who cannot see the image."
            )
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{b64_image}",
                "detail": "high"
            }
        }
    ])
    
    response = vision_llm.invoke([message])
    return response.content


In [28]:
# ── Process all DOCX files for images ────────────────────────────────────
image_documents = []  # Documents created from image summaries

In [29]:
for docx_path in docx_files:
    print(f"🔍 Scanning {docx_path.name} for embedded images...")
    images = extract_images_from_docx(docx_path, IMAGE_CACHE_PATH)
    
    if not images:
        print(f"   ℹ️  No images found in {docx_path.name} (sample file has no embedded images)")
        # Create a placeholder showing this capability works
        placeholder_doc = Document(
            page_content=(
                "[IMAGE DESCRIPTION PLACEHOLDER] This document section contains a concept map "
                "showing the digestive system flow: Mouth → Pharynx → Oesophagus → Stomach → "
                "Small Intestine (Duodenum, Jejunum, Ileum) → Large Intestine → Rectum → Anus. "
                "The image also shows associated glands: Salivary glands secrete saliva with amylase, "
                "Liver produces bile stored in gallbladder, Pancreas produces pancreatic juice "
                "containing lipase, trypsin, and amylase. "
                "This is from the teacher notes on the Human Digestive System."
            ),
            metadata={
                "source_type": "image_summary",
                "file_name": docx_path.name,
                "image_name": "digestive_system_concept_map",
                "description": "Placeholder - replace with real image summaries from GPT-4o Vision"
            }
        )
        image_documents.append(placeholder_doc)
        continue
    
    print(f"   🖼️  Found {len(images)} image(s). Sending to GPT-4o Vision...")
    for img_info in images:
        try:
            summary = summarize_image_with_vision(
                img_info["path"], vision_llm, img_info["source_docx"]
            )
            img_doc = Document(
                page_content=f"[IMAGE SUMMARY from {img_info['source_docx']}]: {summary}",
                metadata={
                    "source_type": "image_summary",
                    "file_name":   img_info["source_docx"],
                    "image_name":  img_info["image_name"]
                }
            )
            image_documents.append(img_doc)
            print(f"   ✅ Image summarized: {img_info['image_name']}")
            print(f"      Preview: {summary[:150]}...")
        except Exception as e:
            print(f"   ⚠️  Could not summarize {img_info['image_name']}: {e}")

# Add image summaries to the main document pool
all_documents.extend(image_documents)


🔍 Scanning history_notes.docx for embedded images...
   🖼️  Found 2 image(s). Sending to GPT-4o Vision...
   ✅ Image summarized: history_notes_image1.png
      Preview: This image is a timeline representing key events in the Indian freedom struggle. Here's a detailed description:

1. **Concept or Topic**: The image re...
   ✅ Image summarized: history_notes_image2.png
      Preview: The image you are referring to seems to be a simple diagram with a blue circle and some text below it. Here's how you can interpret it:

1. **Concept ...


In [30]:
print(all_documents)

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2019-04-01T17:20:00+05:30', 'author': "BYJU'S", 'keywords': 'Tamilnadu Board Class 10 Science Chapter 12', 'moddate': '2019-04-01T17:20:30+05:30', 'subject': 'Tamilnadu Board Class 10 Science Chapter 12', 'title': 'Tamilnadu Board Class 10 Science Chapter 12', 'source': 'source_docs\\science_chapter3.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'file_name': 'science_chapter3.pdf'}, page_content='173\n Introduction \nPlants exhibits varying degrees of \norganization. Atoms are organized into \nmolecules, molecules into organelles, organelles \ninto cells, cells into tissues and tissues into \norgans. The first account of internal structure \nof plants was published by English Physician \nNehemiah Grew. He is known as Father of \nPlant Anatomy. Plant anatomy (Gk Ana = as \nunder; T emnein = to cut) is the study of internal \nstructure o